<a href="https://colab.research.google.com/github/HariniAnandkumar/neuromorphic-ai-research/blob/main/02_snn_fmnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import PyTorch library for deep learning operations
import torch

# Check whether CUDA (GPU acceleration) is available
# GPU training is significantly faster than CPU training
print("CUDA available:", torch.cuda.is_available())

# Print GPU name if available
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU name: Tesla T4


In [ ]:
# Install snnTorch library for building Spiking Neural Networks
!pip install snntorch -q

# PyTorch neural network module
import torch.nn as nn

# Optimization algorithms such as Adam
import torch.optim as optim

# torchvision provides datasets like MNIST
import torchvision

# Image preprocessing utilities
import torchvision.transforms as transforms

# snnTorch library for spiking neurons
import snntorch as snn

# spikegen contains spike encoding methods
from snntorch import spikegen

# Used for measuring training and inference time
import time

# Used for saving results into tables
import pandas as pd

# Set random seeds for reproducibility
# This helps experiments produce consistent results across runs
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Automatically use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 12.1 MB/s eta 0:00:00
Device: cuda


In [ ]:
# Convert MNIST images into tensors
# Neural networks require tensor inputs instead of PIL images
transform = transforms.ToTensor()

# Load MNIST training dataset
# train=True loads 60,000 training images
trainset = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Load MNIST testing dataset
# train=False loads 10,000 unseen test images
testset = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# DataLoader splits data into batches
# batch_size=128 means 128 images are processed together
# shuffle=True randomizes training order each epoch
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=128,
    shuffle=True
)

# Test data is not shuffled because order does not matter during evaluation
testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=128,
    shuffle=False
)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.70MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 130kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.21MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.3MB/s]


In [ ]:
# Number of simulation time steps
# The same image is processed repeatedly over time
# allowing neurons to accumulate membrane potential
num_steps = 10

# Membrane decay constant for LIF neurons
# Higher beta means neurons remember past inputs longer
beta = 0.95

# Define Spiking Neural Network architecture
class SNN(nn.Module):

    def __init__(self):
        super().__init__()

        # First convolution layer
        # Input: grayscale image
        # Output: 32 feature maps
        self.conv1 = nn.Conv2d(1, 32, 3)

        # LIF spiking neuron layer
        self.lif1 = snn.Leaky(beta=beta)

        # Second convolution layer
        self.conv2 = nn.Conv2d(32, 64, 3)

        # Second LIF layer
        self.lif2 = snn.Leaky(beta=beta)

        # Reduce spatial dimensions
        self.pool = nn.MaxPool2d(2)

        # Convert 2D feature maps into 1D vector
        self.flatten = nn.Flatten()

        # Fully connected hidden layer
        self.fc1 = nn.Linear(9216, 128)

        # Third LIF layer
        self.lif3 = snn.Leaky(beta=beta)

        # Output classification layer
        self.fc2 = nn.Linear(128, 10)

        # Final LIF layer
        self.lif4 = snn.Leaky(beta=beta)


    # Forward pass through network
    def forward(self, x):

        # Initialize membrane potentials for all LIF layers
        # Membrane potential stores accumulated neuron activity over time
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()

        # Lists used to record spike activity over time
        spk1_rec = []
        spk2_rec = []
        spk3_rec = []
        spk4_rec = []


         # Run network for multiple time steps
        # At each step neurons integrate inputs and generate spikes
        for step in range(num_steps):

            # First convolution operation
            cur1 = self.conv1(x)

            # LIF neuron updates membrane potential and generates spikes
            spk1, mem1 = self.lif1(cur1, mem1)

            # Second convolution operation
            cur2 = self.conv2(spk1)

            # Second spiking layer
            spk2, mem2 = self.lif2(cur2, mem2)

            # Downsample feature maps
            pooled = self.pool(spk2)

            # Flatten feature maps into vector
            flat = self.flatten(pooled)

            # First fully connected layer
            cur3 = self.fc1(flat)

            # Third spiking layer
            spk3, mem3 = self.lif3(cur3, mem3)

            # Final classification layer
            cur4 = self.fc2(spk3)

            # Output spiking layer
            spk4, mem4 = self.lif4(cur4, mem4)


            # Store spike activity from each layer
            # Used later for spike analysis and counting
            spk1_rec.append(spk1)
            spk2_rec.append(spk2)
            spk3_rec.append(spk3)
            spk4_rec.append(spk4)


        # Stack spike recordings across time steps
        return (
            torch.stack(spk1_rec),
            torch.stack(spk2_rec),
            torch.stack(spk3_rec),
            torch.stack(spk4_rec)
        )

# Create model and move it to GPU/CPU
model = SNN().to(device)

print("Model created successfully")

Model created successfully


In [ ]:
# CrossEntropyLoss is used for multi-class classification
criterion = nn.CrossEntropyLoss()

# Adam optimizer updates model weights during training
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Learning rate scheduler gradually lowers learning rate
# This helps training stabilize during later epochs
from torch.optim.lr_scheduler import StepLR

scheduler = StepLR(
    optimizer,
    step_size=5,
    gamma=0.5
)

# Number of full passes through training dataset
epochs = 15

# Start timer for measuring total training time
start_time = time.time()

# Training loop
for epoch in range(epochs):

    # Set model to training mode
    model.train()

    running_loss = 0

    # Loop through training batches
    for images, labels in trainloader:

        # Move images and labels to GPU/CPU
        images, labels = images.to(device), labels.to(device)

        # Reset gradients before each update
        optimizer.zero_grad()

        # Forward pass through SNN
        spk1, spk2, spk3, spk4 = model(images)

        # Sum output spikes across all time steps
        # The class with highest spike activity becomes prediction
        output_sum = spk4.sum(dim=0)

        # Compute classification loss
        loss = criterion(output_sum, labels)

        # Compute gradients
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

    # Update learning rate scheduler
    scheduler.step()

    # ---------------- EVALUATION AFTER EACH EPOCH ----------------

    model.eval()

    correct = 0
    total = 0

    # Disable gradients during testing
    with torch.no_grad():

        for images, labels in testloader:

            images, labels = images.to(device), labels.to(device)

            # Forward pass
            spk1, spk2, spk3, spk4 = model(images)

            # Sum spikes across time
            output_sum = spk4.sum(dim=0)

            # Predicted class = highest spike activity
            _, predicted = torch.max(output_sum, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    # Compute test accuracy
    epoch_acc = 100 * correct / total

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {running_loss/len(trainloader):.4f} | "
        f"Test Acc: {epoch_acc:.2f}%"
    )

# Compute total training time
train_time = time.time() - start_time

print("Training Time (sec):", train_time)

Epoch 1/15 | Loss: 0.1500 | Test Acc: 98.28%
Epoch 2/15 | Loss: 0.0422 | Test Acc: 98.43%
Epoch 3/15 | Loss: 0.0256 | Test Acc: 98.85%
Epoch 4/15 | Loss: 0.0169 | Test Acc: 98.83%
Epoch 5/15 | Loss: 0.0122 | Test Acc: 98.67%
Epoch 6/15 | Loss: 0.0072 | Test Acc: 99.07%
Epoch 7/15 | Loss: 0.0045 | Test Acc: 99.09%
Epoch 8/15 | Loss: 0.0038 | Test Acc: 99.11%
Epoch 9/15 | Loss: 0.0031 | Test Acc: 99.17%
Epoch 10/15 | Loss: 0.0035 | Test Acc: 99.16%
Epoch 11/15 | Loss: 0.0023 | Test Acc: 99.14%
Epoch 12/15 | Loss: 0.0020 | Test Acc: 99.16%
Epoch 13/15 | Loss: 0.0019 | Test Acc: 99.17%
Epoch 14/15 | Loss: 0.0018 | Test Acc: 99.16%
Epoch 15/15 | Loss: 0.0016 | Test Acc: 99.16%
Training Time (sec): 931.3196320533752


In [ ]:
# ---------------- FINAL EVALUATION ----------------

model.eval()

correct = 0
total = 0

# Per-layer spike + slot accumulators (4 spiking layers)
spk_totals = [0.0, 0.0, 0.0, 0.0]
slots      = [0,   0,   0,   0]

with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        spk1, spk2, spk3, spk4 = model(images)
        spks = [spk1, spk2, spk3, spk4]

        output_sum = spk4.sum(dim=0)
        _, predicted = torch.max(output_sum, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        for i, s in enumerate(spks):
            spk_totals[i] += s.sum().item()
            slots[i]      += s.numel()

test_acc = 100 * correct / total

# ---- SynOps + spike rate ----
# fanout = synapses each spiking neuron drives into the NEXT layer.
# lif1 (32ch) feeds conv2 = Conv2d(32,64,3) -> 64*3*3 = 576
# lif2 (64ch) feeds fc1 (dense, 128 wide)   -> 128  (pooling treated as routing)
# lif3 (128)  feeds fc2 = Linear(128,10)    -> 10
# lif4 (10)   is output, drives nothing     -> 0
fanouts = [576, 128, 10, 0]

total_spikes = sum(spk_totals)
synops = sum(spk_totals[i] * fanouts[i] for i in range(4))
spike_rate = total_spikes / sum(slots)

print("Test Accuracy:", test_acc)
print("Total Spikes:", total_spikes)
print("SynOps:", synops)
print("SynOps per image:", synops / total)
print("Spike Rate (avg fraction firing):", spike_rate)

Test Accuracy: 99.16
Total Spikes: 219124937.0
SynOps: 106997196726.0
SynOps per image: 10699719.6726
Spike Rate (avg fraction firing): 0.03737165074871235


In [ ]:
# ---------------- LATENCY MEASUREMENT ----------------

dummy_input = torch.randn(1, 1, 28, 28).to(device)

# Warmup — first GPU calls are always slow, exclude them
for _ in range(10):
    with torch.no_grad():
        _ = model(dummy_input)
if device.type == "cuda":
    torch.cuda.synchronize()

# Average over 100 runs for a stable number
start = time.time()
n_runs = 100
with torch.no_grad():
    for _ in range(n_runs):
        _ = model(dummy_input)
if device.type == "cuda":
    torch.cuda.synchronize()
latency = (time.time() - start) / n_runs

print("Avg Inference Latency (sec):", latency)

# ---------------- SAVE RESULTS ----------------

metrics = {
    "model": "SNN",
    "accuracy": test_acc,
    "train_time_sec": train_time,
    "latency_sec": latency,
    "params": sum(p.numel() for p in model.parameters()),
    "synops": synops,
    "synops_per_image": synops / total,
    "spike_rate": spike_rate,
    "total_spikes": total_spikes,
}

df = pd.DataFrame([metrics])
df.to_csv("snn_fmnist_results.csv", index=False)
print("Saved snn_fmnist_results.csv")

torch.save(model.state_dict(), "snn_fmnist.pth")
print("Saved snn_fmnist.pth")
print("Saved snn_mnist.pth")

Avg Inference Latency (sec): 0.01781724214553833
Saved snn_fmnist_results.csv
Saved snn_fmnist.pth
Saved snn_mnist.pth
